In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, PULP_CBC_CMD
import numpy as np
import unicodedata
import difflib

print("=" * 60)
print("🏗️ STEP 1: RAW-TO-RAW ENTITY ALIGNMENT ENGINE")
print("=" * 60)

predictions_path = Path("../data/processed/world_cup_predictions_adjusted.csv")
prices_path = Path("../scrapers/fifa_fantasy/output/fantasy_prices.csv")
audit_output_path = Path("../data/processed/matching_audit.csv")
output_pool_path = Path("../data/processed/master_player_pool.csv")

df_pred = pd.read_csv(predictions_path)
df_prices = pd.read_csv(prices_path)

# 🎯 INTUITIVE DICTIONARY: "Raw Name in Predictions" : "Raw Name in Fantasy Prices (Short or Full)"
# You can now paste names EXACTLY as they appear in your CSV files.
MANUAL_MAPPING = {
    "kylian mbappe": "mbappe",
    "vinicius junior": "vinícius júnior",
    "neymar jr": "neymar",
    "patrik schick": "schick",
    "Antonio RUEDIGER" : "Antonio Rüdiger",
    "Eray COEMERT":"Eray Cömert",
    "DANILO SANTOS":"Danilo dos Santos de Oliveira",
    "CRISTIANO RONALDO":"Cristiano Ronaldo dos Santos Aveiro",
    "Adam AROUS":"Adem Arous",
    "ALI ALHAMADI":"Ali Ibrahim Karim Ali Al Hamadi",
    "ROGER IBANEZ":"Roger Ibañez da Silva",
    "Phillip MWENE":"Phillipp Mwene",
    "FRANCISCO TRINCAO":"Francisco António Machado Mota de Castro Trincão",
    "Alessandro SCHOEPF":"Alessandro Schöpf",
    "Torbjorn HEGGEM":"Torbjørn Heggem",
    "Pascal GROSS":"Pascal Groß",
    "MAURICIO":"Maurício Magalhães Prado",
    "Alexander NUEBEL":"Alexander Nübel",
    "Alexander SORLOTH":"Alexander Sørloth",
    "Orjan NYLAND":"Ørjan Nyland",
    "Yassine TITRAOUI":"Yacine Titraoui",
    "Aiden ONEILL":"Aiden O'Neill"
}

# Apply manual overrides directly to the raw names before any text mutation
df_pred['mapped_name'] = df_pred['name'].map(lambda x: MANUAL_MAPPING.get(x, x))

def normalize_text(text):
    if not isinstance(text, str): 
        return ""
    # Strip accents and diacritics
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    # Clean punctuation and lowercase
    text = text.lower().replace('.', '').replace('-', ' ').replace(',', '').strip()
    # Sort words alphabetically to handle swapped first/last name formats
    return " ".join(sorted(text.split()))

# Generate lookup keys using the normalized, mapped variations
df_pred['lookup_key'] = df_pred['mapped_name'].apply(normalize_text)
df_prices['norm_short'] = df_prices['name'].apply(normalize_text)
df_prices['norm_full'] = df_prices['full_name'].apply(normalize_text)

# Initialize Tracking Columns with explicit string-supporting object types
df_pred['matched_price_name'] = pd.Series(np.nan, index=df_pred.index, dtype=object)
df_pred['price'] = np.nan
df_pred['match_type'] = "Unmatched"

# Map out reference pools for fast dictionary scanning
price_pool_short = dict(zip(df_prices['norm_short'], zip(df_prices['name'], df_prices['price'])))
price_pool_full = dict(zip(df_prices['norm_full'], zip(df_prices['full_name'], df_prices['price'])))

print("Executing Multi-Tier Matching Sequence...")

for idx, row in df_pred.iterrows():
    key = row['lookup_key']
    
    # Tier 1: Match against short name variations
    if key in price_pool_short:
        df_pred.at[idx, 'matched_price_name'] = price_pool_short[key][0]
        df_pred.at[idx, 'price'] = price_pool_short[key][1]
        df_pred.at[idx, 'match_type'] = "1_Exact_Short"
        continue
        
    # Tier 2: Match against full name variations
    if key in price_pool_full:
        df_pred.at[idx, 'matched_price_name'] = price_pool_full[key][0]
        df_pred.at[idx, 'price'] = price_pool_full[key][1]
        df_pred.at[idx, 'match_type'] = "2_Exact_Full"
        continue
        
    # Tier 3: Context-aware fuzzy string lookup fallback
    matches = difflib.get_close_matches(key, list(price_pool_short.keys()), n=1, cutoff=0.70)
    if matches:
        best_match = matches[0]
        df_pred.at[idx, 'matched_price_name'] = price_pool_short[best_match][0]
        df_pred.at[idx, 'price'] = price_pool_short[best_match][1]
        df_pred.at[idx, 'match_type'] = "3_Fuzzy"

# Generate Audit Report Matrix
audit_df = df_pred[['name', 'country', 'position', 'adjusted_projection', 'matched_price_name', 'price', 'match_type']].copy()
audit_df.sort_values(by='match_type', inplace=True)
audit_df.to_csv(audit_output_path, index=False)

print(f"\n✅ Alignment complete. Audit ledger updated: {audit_output_path.name}")
print("\n📊 MATCHING STATISTICS:")
print("-" * 40)
print(df_pred['match_type'].value_counts())
print("-" * 40)

print("\n" + "=" * 60)
print("🛡️ STEP 2: DATA INTEGRITY VALIDATION GATE")
print("=" * 60)

# Halt execution only if high-value point scorers slip into fallback groups
ELITE_THRESHOLD_POINTS = 5.5
leaked_elites = df_pred[(df_pred['match_type'] == 'Unmatched') & (df_pred['adjusted_projection'] >= ELITE_THRESHOLD_POINTS)]

if not leaked_elites.empty:
    print(f"❌ CRITICAL: {len(leaked_elites)} Elite assets remain unmatched!")
    display(leaked_elites[['name', 'country', 'adjusted_projection']])
    raise AssertionError("Pipeline paused. Update MANUAL_MAPPING dictionary using exact original CSV casing.")

# Safe fallback assignments for lower tier squad depth
unmatched_count = df_pred[df_pred['match_type'] == 'Unmatched'].shape[0]
print(f"📋 Unmatched depth players receiving median price fallback: {unmatched_count}")

median_price = df_prices['price'].median()
# Calculate Fantasy-Specific Gem Score dynamically
df_pred['pred_pct'] = df_pred['adjusted_projection'].rank(pct=True) * 100
df_pred['price_pct'] = df_pred['price'].rank(pct=True) * 100
df_pred['gem_score_adj'] = df_pred['pred_pct'] - df_pred['price_pct']

# Construct sanitized dataframe layout for the optimization engine
df_master_pool = df_pred.rename(columns={'name': 'player'})[['player', 'country', 'position', 'price', 'adjusted_projection', 'std_fp_last_5', 'gem_score_adj']]
df_master_pool.to_csv(output_pool_path, index=False)
print(f"✅ Final Master Pool exported to: {output_pool_path.name}")

🏗️ STEP 1: RAW-TO-RAW ENTITY ALIGNMENT ENGINE
Executing Multi-Tier Matching Sequence...

✅ Alignment complete. Audit ledger updated: matching_audit.csv

📊 MATCHING STATISTICS:
----------------------------------------
match_type
2_Exact_Full     635
1_Exact_Short     89
3_Fuzzy           18
Unmatched          7
Name: count, dtype: int64
----------------------------------------

🛡️ STEP 2: DATA INTEGRITY VALIDATION GATE
📋 Unmatched depth players receiving median price fallback: 7
✅ Final Master Pool exported to: master_player_pool.csv


In [3]:
import pandas as pd
import numpy as np
from pulp import *
from pathlib import Path

print("=" * 60)
print("🧠 INITIALIZING MULTI-OBJECTIVE FANTASY ENGINE")
print("=" * 60)

pool_path = Path("../data/processed/master_player_pool.csv")
df = pd.read_csv(pool_path)

print("🧹 Executing Data Sanitization Protocol...")
df = df.dropna(subset=['player'])

numeric_cols = ['price', 'adjusted_projection', 'std_fp_last_5', 'gem_score_adj']
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

df['price'] = df['price'].fillna(df['price'].median())
df['adjusted_projection'] = df['adjusted_projection'].fillna(0)
df['std_fp_last_5'] = df['std_fp_last_5'].fillna(0)
df['gem_score_adj'] = df['gem_score_adj'].fillna(0)

players = df['player'].tolist()
costs = dict(zip(players, df['price']))
points = dict(zip(players, df['adjusted_projection']))
positions = dict(zip(players, df['position']))
countries = dict(zip(players, df['country']))
volatility = dict(zip(players, df['std_fp_last_5']))
gem_scores = dict(zip(players, df['gem_score_adj']))

print("✅ Data artifacts sanitized and mapped.")

def solve_squad(strategy="meta", previous_squads=[], max_overlap=11):
    prob = LpProblem(f"WC2026_{strategy.capitalize()}", LpMaximize)
    player_vars = LpVariable.dicts("Select", players, cat=LpBinary)

    if strategy == "meta":
        prob += lpSum([points[p] * player_vars[p] for p in players])
    elif strategy == "upside":
        prob += lpSum([(points[p] * (1 + volatility[p])) * player_vars[p] for p in players])
    elif strategy == "value":
        prob += lpSum([((points[p] / costs[p]) + (gem_scores[p] * 0.1)) * player_vars[p] for p in players])

    prob += lpSum([player_vars[p] for p in players]) == 15, "Total_15"
    prob += lpSum([costs[p] * player_vars[p] for p in players]) <= 100.0, "Budget"
    prob += lpSum([player_vars[p] for p in players if positions[p] == 2]) == 2, "GK_Req"
    prob += lpSum([player_vars[p] for p in players if positions[p] == 3]) == 5, "DEF_Req"
    prob += lpSum([player_vars[p] for p in players if positions[p] == 1]) == 5, "MID_Req"
    prob += lpSum([player_vars[p] for p in players if positions[p] == 0]) == 3, "FWD_Req"

    for country in set(countries.values()):
        prob += lpSum([player_vars[p] for p in players if countries[p] == country]) <= 3, f"Max3_{country}"

    for idx, prev_squad in enumerate(previous_squads):
        prob += lpSum([player_vars[p] for p in prev_squad]) <= max_overlap, f"Diversity_Cut_{idx}"

    prob.solve(PULP_CBC_CMD(msg=0))
    
    if prob.status == 1:
        return [p for p in players if player_vars[p].varValue == 1.0]
    return []

print("✅ Optimization constraints built. Running combinatorial algorithms...\n")

squads_data = {}
all_generated_squads = []

strategies = ["meta", "upside", "value"]
titles = {
    "meta": "🛡️ SQUAD 1: THE META (Highest Expected Points)",
    "upside": "🚀 SQUAD 2: HIGH UPSIDE (Tournament Winner Volatility)",
    "value": "💎 SQUAD 3: DIFFERENTIAL VALUE (Points per Million + Gems)"
}

for strat in strategies:
    squad_list = solve_squad(strategy=strat, previous_squads=all_generated_squads, max_overlap=11)
    all_generated_squads.append(squad_list)
    
    df_sq = df[df['player'].isin(squad_list)].copy()
    df_sq = df_sq.sort_values(by=['position', 'price'], ascending=[False, False])
    squads_data[strat] = df_sq
    
    print("=" * 60)
    print(titles[strat])
    print("=" * 60)
    print(df_sq[['player', 'country', 'position', 'price', 'adjusted_projection']].to_string(index=False))
    print(f"\nTotal Cost: €{df_sq['price'].sum():.1f} | Projected Points: {df_sq['adjusted_projection'].sum():.2f}\n")

print("=" * 60)
print("👑 GLOBAL CAPTAINCY MATRIX")
print("=" * 60)

df['safe_cap_score'] = df['adjusted_projection'] / (1 + df['std_fp_last_5'])
df['agg_cap_score'] = df['adjusted_projection'] * (1 + df['std_fp_last_5'])
df['diff_cap_score'] = df['adjusted_projection'] * (1 + (df['gem_score_adj']/100))

print("\n🛡️ TOP 3 SAFE CAPTAINS (High Floor, Low Variance)")
safe = df.sort_values(by='safe_cap_score', ascending=False).head(3)
print(safe[['player', 'country', 'price', 'adjusted_projection']].to_string(index=False))

print("\n🚀 TOP 3 AGGRESSIVE CAPTAINS (High Ceiling, High Volatility)")
agg = df.sort_values(by='agg_cap_score', ascending=False).head(3)
print(agg[['player', 'country', 'price', 'adjusted_projection']].to_string(index=False))

print("\n💎 TOP 3 DIFFERENTIAL CAPTAINS (Hidden Gems + High Output)")
diff = df[df['price'] <= 8.5].sort_values(by='diff_cap_score', ascending=False).head(3)
print(diff[['player', 'country', 'price', 'adjusted_projection']].to_string(index=False))

🧠 INITIALIZING MULTI-OBJECTIVE FANTASY ENGINE
🧹 Executing Data Sanitization Protocol...
✅ Data artifacts sanitized and mapped.
✅ Optimization constraints built. Running combinatorial algorithms...

🛡️ SQUAD 1: THE META (Highest Expected Points)
        player   country  position  price  adjusted_projection
Enzo FERNANDEZ Argentina         3    7.5                 4.55
 Morgan ROGERS   England         3    7.2                 4.19
Ismael SAIBARI   Morocco         3    6.8                 4.38
 LUCAS PAQUETA    Brazil         3    6.5                 3.99
Nicolas RASKIN   Belgium         3    5.3                 4.15
 Senne LAMMENS   Belgium         2    4.6                 4.22
James TRAFFORD   England         2    4.0                 4.50
William SALIBA    France         1    5.3                 4.89
Theo HERNANDEZ    France         1    5.0                 4.69
Thomas MEUNIER   Belgium         1    4.8                 4.79
  Cesar MONTES    Mexico         1    4.7                 4.71